---
## 14. Real-World Practice — The Titanic Dataset

We've been building toward this all lecture: real historical data, 891 passenger records from
the RMS Titanic (1912) — the original, full passenger list, with a `Survived` column (0 = No,
1 = Yes) at the center of it. Every stage we practiced today — Read, Explore, Select, Filter,
Clean, Group, Combine, Transform — applies directly here.

**This section is yours.** The setup below loads the data — from here on, work through the task
list independently, the same way you'll be expected to work on real data going forward.


In [1]:
import pandas as pd
import numpy as np

titanic = pd.read_csv('titanic.csv')
titanic.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


### Dataset Notes

This version has the full original columns, including `PassengerId` (genuine unique key),
`Name`, `Ticket`, and `Cabin` in their raw form.

### Your Tasks
1. **Explore** — `.shape`, `.info()`, `.describe()`, missing value counts
2. **Select** — pick `Survived`, `Pclass`, `Sex`, `Age`, `Fare`
3. **Filter** — female, 1st class, survived
4. **Sort** — by `Fare` descending, top 10
5. **Create column** — `AgeGroup` bucket
6. **Handle missing** — `Age`, `Cabin`, `Embarked` each treated differently
7. **Duplicates** — check & interpret
8. **Group & aggregate** — survival rate by `Sex` and `Pclass`
9. **Pivot table** — `Pclass` x `Sex`
10. **Transform** — extract `Title` from `Name`, survival rate per title


In [2]:
# ============================================================
# Task 1: Explore
# ============================================================

print('Shape (rows, columns):', titanic.shape)


Shape (rows, columns): (891, 12)


In [3]:
# Detailed column types and non-null counts
titanic.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


In [4]:
# Statistical summary for all numeric columns
titanic.describe()


,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.000000,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,446.000000,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,257.353842,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,1.000000,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,223.500000,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,446.000000,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,668.500000,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,891.000000,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


In [5]:
# Missing values — count and percentage per column
missing = titanic.isnull().sum()
missing_pct = (missing / len(titanic) * 100).round(2)

missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct
}).query('`Missing Count` > 0')

print(missing_df)
print()
print('--- Analysis ---')
print('Age      : ~20% missing  -> moderate, imputable with median')
print('Cabin    : ~77% missing  -> extreme, mostly lower-class passengers with no recorded cabin')
print('Embarked : <1%  missing  -> negligible, fill with mode (most common port)')


          Missing Count  Missing %
Age                 177      19.87
Cabin               687      77.10
Embarked              2       0.22

--- Analysis ---
Age      : ~20% missing  -> moderate, imputable with median
Cabin    : ~77% missing  -> extreme, mostly lower-class passengers with no recorded cabin
Embarked : <1%  missing  -> negligible, fill with mode (most common port)


In [6]:
# ============================================================
# Task 2: Select
# ============================================================

titanic_core = titanic[['Survived', 'Pclass', 'Sex', 'Age', 'Fare']].copy()

print('Selected DataFrame shape:', titanic_core.shape)
titanic_core.head(10)


Selected DataFrame shape: (891, 5)


,Survived,Pclass,Sex,Age,Fare
0,0,3,male,22.0,7.2500
1,1,1,female,38.0,71.2833
2,1,3,female,26.0,7.9250
3,1,1,female,35.0,53.1000
4,0,3,male,35.0,8.0500
5,0,3,male,NaN,8.4583
6,0,1,male,54.0,51.8625
7,0,3,male,2.0,21.0750
8,1,3,female,27.0,11.1333
9,1,2,female,14.0,30.0708


In [7]:
# ============================================================
# Task 3: Filter — female, 1st class, survived
# ============================================================

mask = (
    (titanic['Sex'] == 'female') &
    (titanic['Pclass'] == 1) &
    (titanic['Survived'] == 1)
)

survivors_1f = titanic[mask][['Name', 'Age', 'Fare', 'Cabin', 'Embarked']]
print(f'Total 1st-class female survivors: {len(survivors_1f)}')
survivors_1f.reset_index(drop=True)


Total 1st-class female survivors: 91


,Name,Age,Fare,Cabin,Embarked
0,"Cumings, Mrs. John Bradley (Florence Briggs Th...",38.0,71.2833,C85,C
1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",35.0,53.1000,C123,S
2,"Bonnell, Miss. Elizabeth",58.0,26.5500,C103,S
3,"Spencer, Mrs. William Augustus (Marie Eugenie)",NaN,146.5208,B78,C
4,"Harper, Mrs. Henry Sleeper (Myna Haxtun)",49.0,76.7292,D33,C
...,...,...,...,...,...
86,"Wick, Mrs. George Dennick (Mary Hitchcock)",45.0,164.8667,NaN,S
87,"Swift, Mrs. Frederick Joel (Margaret Welles Ba...",48.0,25.9292,D17,S
88,"Beckwith, Mrs. Richard Leonard (Sallie Monypeny)",47.0,52.5542,D35,S
89,"Potter, Mrs. Thomas Jr (Lily Alexenia Wilson)",56.0,83.1583,C50,C


In [8]:
# ============================================================
# Task 4: Sort by Fare (highest first) — Top 10
# ============================================================

top10 = (
    titanic
    .sort_values('Fare', ascending=False)
    .head(10)
    [['Name', 'Pclass', 'Sex', 'Age', 'Fare', 'Cabin', 'Survived']]
    .reset_index(drop=True)
)
top10


,Name,Pclass,Sex,Age,Fare,Cabin,Survived
0,"Cardeza, Mr. Thomas Drake Martinez",1,male,36.0,512.3292,B51 B53 B55,1
1,"Ward, Miss. Anna",1,female,35.0,512.3292,NaN,1
2,"Lesurer, Mr. Gustave J",1,male,35.0,512.3292,B101,1
3,"Fortune, Miss. Mabel Helen",1,female,23.0,263.0000,C23 C25 C27,1
4,"Fortune, Mr. Mark",1,male,64.0,263.0000,C23 C25 C27,0
5,"Fortune, Miss. Alice Elizabeth",1,female,24.0,263.0000,C23 C25 C27,1
6,"Fortune, Mr. Charles Alexander",1,male,19.0,263.0000,C23 C25 C27,0
7,"Ryerson, Miss. Susan Parker ""Suzette""",1,female,21.0,262.3750,B57 B59 B63 B66,1
8,"Ryerson, Miss. Emily Borie",1,female,18.0,262.3750,B57 B59 B63 B66,1
9,"Baxter, Mrs. James (Helene DeLaudeniere Chaput)",1,female,50.0,247.5208,B58 B60,1


In [9]:
# ============================================================
# Task 5: Create AgeGroup column
#
# Choice for NaN Age: label as 'Unknown'
# Reason: avoids imputation bias at this exploratory stage;
#         we handle NaNs properly in Task 6.
# ============================================================

def age_group(age):
    if pd.isna(age):
        return 'Unknown'
    elif age < 13:
        return 'Child'
    elif age <= 19:
        return 'Teen'
    elif age <= 59:
        return 'Adult'
    else:
        return 'Senior'

titanic['AgeGroup'] = titanic['Age'].apply(age_group)

print('AgeGroup distribution:')
print(titanic['AgeGroup'].value_counts())
print()
titanic[['Name', 'Age', 'AgeGroup']].head(15)


AgeGroup distribution:
AgeGroup
Adult      524
Unknown    177
Teen        95
Child       69
Senior      26
Name: count, dtype: int64



,Name,Age,AgeGroup
0,"Braund, Mr. Owen Harris",22.0,Adult
1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",38.0,Adult
2,"Heikkinen, Miss. Laina",26.0,Adult
3,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",35.0,Adult
4,"Allen, Mr. William Henry",35.0,Adult
5,"Moran, Mr. James",NaN,Unknown
6,"McCarthy, Mr. Timothy J",54.0,Adult
7,"Palsson, Master. Gosta Leonard",2.0,Child
8,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",27.0,Adult
9,"Nasser, Mrs. Nicholas (Adele Achem)",14.0,Teen


In [10]:
# ============================================================
# Task 6: Handle Missing Values
# Three columns — three different strategies
# ============================================================

print('BEFORE — missing per column:')
print(titanic.isnull().sum()[titanic.isnull().sum() > 0])
print()

titanic_clean = titanic.copy()

# Age (~20% missing): fill with MEDIAN
# Why median? Robust to right skew from high-fare 1st-class outliers.
age_median = titanic_clean['Age'].median()
n_age = int(titanic_clean['Age'].isnull().sum())
titanic_clean['Age'] = titanic_clean['Age'].fillna(age_median)
print(f'Age    -> filled {n_age} NaNs with median = {age_median}')

# Cabin (~77% missing): DROP the column
# Why drop? 77% missing makes any imputation unreliable.
# We preserve a binary has_cabin flag instead.
titanic_clean['has_cabin'] = titanic_clean['Cabin'].notna().astype(int)
titanic_clean = titanic_clean.drop(columns=['Cabin'])
print('Cabin  -> column dropped (77% missing); binary has_cabin flag added')

# Embarked (<1% missing): fill with MODE
# Why mode? Only 2 rows missing — trivially filled with most common port.
mode_port = titanic_clean['Embarked'].mode()[0]
titanic_clean['Embarked'] = titanic_clean['Embarked'].fillna(mode_port)
print(f'Embarked -> filled 2 NaNs with mode = "{mode_port}" (Southampton)')

print()
print('AFTER — missing per column:')
remaining = titanic_clean.isnull().sum()
still_missing = remaining[remaining > 0]
if len(still_missing) == 0:
    print('No missing values remaining!')
else:
    print(still_missing)


BEFORE — missing per column:
Age         177
Cabin       687
Embarked      2
dtype: int64

Age    -> filled 177 NaNs with median = 28.0
Cabin  -> column dropped (77% missing); binary has_cabin flag added
Embarked -> filled 2 NaNs with mode = "S" (Southampton)

AFTER — missing per column:
No missing values remaining!


In [11]:
# ============================================================
# Task 7: Check for Duplicates
# ============================================================

dup_rows = titanic.duplicated().sum()
dup_ids  = titanic['PassengerId'].duplicated().sum()

print(f'Fully duplicate rows  : {dup_rows}')
print(f'Duplicate PassengerIds: {dup_ids}')
print()
print('--- Interpretation ---')
print('PassengerId is a declared unique key (one row = one real passenger).')
print('We EXPECT duplicated().sum() == 0.')
print('A non-zero result would mean the CSV was corrupted, merged badly,')
print('or the same passenger record was accidentally entered twice.')
print()
if dup_rows == 0:
    print('Result: CLEAN — no duplicates found. Data integrity confirmed.')
else:
    print(f'WARNING: {dup_rows} duplicate rows detected — investigate before proceeding!')


Fully duplicate rows  : 0
Duplicate PassengerIds: 0

--- Interpretation ---
PassengerId is a declared unique key (one row = one real passenger).
We EXPECT duplicated().sum() == 0.
A non-zero result would mean the CSV was corrupted, merged badly,
or the same passenger record was accidentally entered twice.

Result: CLEAN — no duplicates found. Data integrity confirmed.


In [12]:
# ============================================================
# Task 8: Group & Aggregate — Survival Rate by Sex and Pclass
# ============================================================

surv_sex = (
    titanic.groupby('Sex')['Survived']
    .mean()
    .rename('Survival Rate')
    .mul(100).round(1)
)
print('Survival Rate by Sex:')
print(surv_sex.to_string())
print()

surv_pclass = (
    titanic.groupby('Pclass')['Survived']
    .mean()
    .rename('Survival Rate')
    .mul(100).round(1)
)
print('Survival Rate by Pclass:')
print(surv_pclass.to_string())
print()
print('--- Pattern Observed ---')
print('SEX    : Women ~74% vs Men ~19%  -> "Women and children first" clearly enforced.')
print('PCLASS : 1st ~63% > 2nd ~47% > 3rd ~24% -> Class/wealth drove lifeboat access.')


Survival Rate by Sex:
Sex
female    74.2
male      18.9

Survival Rate by Pclass:
Pclass
1    63.0
2    47.3
3    24.2

--- Pattern Observed ---
SEX    : Women ~74% vs Men ~19%  -> "Women and children first" clearly enforced.
PCLASS : 1st ~63% > 2nd ~47% > 3rd ~24% -> Class/wealth drove lifeboat access.


In [13]:
# ============================================================
# Task 9: Pivot Table — Survival Rate by Pclass x Sex
# ============================================================

pivot = titanic.pivot_table(
    values='Survived',
    index='Pclass',
    columns='Sex',
    aggfunc='mean'
).round(3)

print('Survival Rate Pivot Table (rows = Pclass, cols = Sex):')
pivot


Survival Rate Pivot Table (rows = Pclass, cols = Sex):


Sex,female,male
Pclass,,
1,0.968,0.369
2,0.921,0.157
3,0.500,0.135


In [14]:
# ============================================================
# Task 10: Extract Title from Name
#
# Name format: 'Last, Title. First Middle'
# Strategy   : split on comma -> take [1] -> split on dot -> take [0] -> strip
# ============================================================

titanic['Title'] = (
    titanic['Name']
    .str.split(',').str[1]   # everything after the comma
    .str.split('.').str[0]   # everything before the first dot
    .str.strip()             # remove surrounding whitespace
)

print('Unique titles found:')
print(titanic['Title'].value_counts())


Unique titles found:
Title
Mr              517
Miss            182
Mrs             125
Master           40
Dr                7
Rev               6
Col               2
Mlle              2
Major             2
Ms                1
Mme               1
Don               1
Lady              1
Sir               1
Capt              1
the Countess      1
Jonkheer          1
Name: count, dtype: int64


In [15]:
# Survival Rate per Title
# ============================================================

title_survival = (
    titanic.groupby('Title')['Survived']
    .agg(['mean', 'count'])
    .rename(columns={'mean': 'Survival Rate', 'count': 'Count'})
    .sort_values('Survival Rate', ascending=False)
    .round(3)
)

print('Survival Rate per Title:')
print(title_survival.to_string())
print()
print('--- Insight ---')
print('Title reveals more nuance than Sex alone:')
print('  Mrs.    (~80%): married women -> highest survival group')
print('  Miss.   (~70%): unmarried women/girls -> high survival')
print('  Master. (~57%): young boys -> benefited from "children first"')
print('  Mr.     (~16%): adult men -> lowest survival by far')
print('  Rare titles (Dr., Rev., Col.): edge cases, low counts')
print()
print('Conclusion: Title encodes both sex AND age in one field,')
print('giving richer predictive signal than Sex column alone.')


Survival Rate per Title:
              Survival Rate  Count
Title                             
Lady                  1.000      1
Ms                    1.000      1
Sir                   1.000      1
Mme                   1.000      1
the Countess          1.000      1
Mlle                  1.000      2
Mrs                   0.792    125
Miss                  0.698    182
Master                0.575     40
Major                 0.500      2
Col                   0.500      2
Dr                    0.429      7
Mr                    0.157    517
Capt                  0.000      1
Jonkheer              0.000      1
Don                   0.000      1
Rev                   0.000      6

--- Insight ---
Title reveals more nuance than Sex alone:
  Mrs.    (~80%): married women -> highest survival group
  Miss.   (~70%): unmarried women/girls -> high survival
  Master. (~57%): young boys -> benefited from "children first"
  Mr.     (~16%): adult men -> lowest survival by far
  Rare titles (Dr.